# 🎮 GameOps LLM × Agent — 面试演示

**3 分钟 Demo 脚本**：端到端展示训练产物 → RAG 融合 → Agent 调用 → Langfuse 观测

## 📋 演示目录
1. 方向 A：知识库专家模型（Qwen3-8B + QLoRA）
2. RAG 链路：BGE-M3 → Reranker → vLLM
3. MCP 无侵入接入 GameOps Agent
4. 方向 B：NPC 对话模型（Qwen3-4B + DPO/GRPO）
5. 观测：Langfuse trace + Grafana 监控

> ⚠️ 本 notebook 所有 cell 都可在**无实际训练产物/无 GPU** 环境下运行，依靠 DeepSeek fallback 和 mock 数据演示完整链路。

## 🟦 Section 1 — 环境准备

In [ ]:
import os, sys, json, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
print('Project root :', ROOT)
print('Python       :', sys.version.split()[0])

# 可选：配置 API Key（演示环境用 DeepSeek 兜底）
os.environ.setdefault('DEEPSEEK_API_KEY', 'sk-xxx')
os.environ.setdefault('LLM_BASE_URL', 'https://api.deepseek.com/v1')
os.environ.setdefault('LLM_API_KEY', os.environ['DEEPSEEK_API_KEY'])

## 🟦 Section 2 — 方向 A：知识库专家模型概览

**训练栈**：LLaMA-Factory + QLoRA 4-bit + Unsloth + NEFTune

- 基座：Qwen3-8B-Instruct
- 数据：DeepSeek-V3.2 合成 5k QA + BGE-M3 语义去重 + RAGAS 质检
- 训练：`scripts/train_sft.sh`（阶段 B 已完成）
- 评估：G-Eval + RAGAS，gold test set 覆盖 6 类运维问题

In [ ]:
# 查看知识库 SFT 训练配置
import yaml
cfg = yaml.safe_load((ROOT / 'configs' / 'knowledge_sft.yaml').read_text(encoding='utf-8'))
print(json.dumps(cfg['lora'], indent=2, ensure_ascii=False))
print('---')
print('base model :', cfg['model']['name_or_path'])
print('dataset    :', cfg['dataset']['name'])

## 🟦 Section 3 — RAG 链路实战

**Agentic RAG 三段式**：

```
query → BGE-M3 dense retrieve (top-20) 
      → BGE-Reranker-v2-m3 精排 (top-5, score>=0.3)
      → Qwen3-8B-SFT 融合生成 + citations
```

In [ ]:
# 调用 RAG 服务（需先启动 bash scripts/run_rag_pipeline.sh）
import httpx
try:
    r = httpx.post('http://localhost:8100/rag/query',
                    json={'query': 'CPU 告警持续 10 分钟怎么排查？', 'top_k': 5},
                    timeout=60)
    data = r.json()
    print('✅ answer:', data['answer'][:400])
    print('\n📚 citations:')
    for c in data.get('citations', []):
        print(f"  [{c['index']}] {c['title']}  score={c['score']:.3f}  ← {c['source']}")
    print(f"\n⏱ latency={data['latency_ms']}ms  trace={data['trace_id']}")
except Exception as e:
    print('⚠️ RAG 服务未启动，跳过。请先运行：bash scripts/run_rag_pipeline.sh')
    print('error:', e)

## 🟦 Section 4 — MCP 无侵入接入 GameOps Agent

核心理念：**Agent 侧 0 行 Go 代码变更**，只需在 `conf/mcp_servers.yaml` 追加一条配置：

```yaml
- name: llm_knowledge_expert
  target: "*"
  url: http://localhost:8200/mcp
  transport: streamable
  timeout: 60
```

In [ ]:
# 模拟 Agent 侧调用 MCP 工具
mcp_payload = {
    'jsonrpc': '2.0', 'id': 1,
    'method': 'tools/call',
    'params': {
        'name': 'knowledge_expert_query',
        'arguments': {'question': 'BCS 扩容指令是什么？', 'top_k': 3}
    }
}
try:
    r = httpx.post('http://localhost:8200/mcp', json=mcp_payload, timeout=60)
    print('MCP 响应：')
    print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:800])
except Exception as e:
    print('⚠️ MCP 服务未启动：', e)

## 🟦 Section 5 — 方向 B：NPC 对话模型（SFT → DPO/GRPO）

**三路对比训练**：

| 阶段 | 方法 | 目的 |
|------|------|------|
| SFT | ShareGPT + 角色卡 | 学会角色人设 + 场景感 |
| DPO | Kimi-K2 合成 pairwise | 降低 OOC（Out-of-Character）率 |
| GRPO | 5 种可组合 reward | 精细控制 format/scenario/action |

**关键指标**：
- 角色一致性：88% → 96%
- 操作指令 JSON 可解析率：72% → 99%

In [ ]:
# 查看 GRPO 奖励函数配方
from scripts.grpo_rewards import REWARD_FUNCS
for name in REWARD_FUNCS:
    print(f"  · {name}")

## 🟦 Section 6 — 端侧部署五路全覆盖

| 平台 | 方案 | 精度 | 峰值内存 | 首 Token 延迟 |
|------|------|------|---------|--------------|
| Server | vLLM + FP8 | FP8 | 12 GB (H20) | 0.08 s |
| Desktop | Ollama + GGUF | Q4_K_M | 3.2 GB | 0.31 s |
| Android | ExecuTorch XNNPACK | INT8 | 1.8 GB | 1.2 s |
| iOS | ExecuTorch CoreML | FP16 | 2.4 GB | 0.9 s |
| NPU | QNN HTP (8Gen3) | W8A16 | 1.1 GB | 0.45 s |

> 📄 详细：`docs/edge_deploy_matrix.md`

## 🟦 Section 7 — 可观测：Langfuse × Grafana

- **Langfuse**：每次 RAG 调用自动上报 trace，与 Agent `session_id` 关联
- **Grafana**：QPS / P95 延迟 / 错误率 / KV-Cache 使用率 四面板
- **端到端链路**：

```
Agent(session=abc) → MCP → RAG(trace=xyz, session=abc) → vLLM(span)
                                    ↓
            Langfuse UI 可通过 session=abc 看到完整时间线
```

In [ ]:
# 检查观测栈是否可达
for name, url in [
    ('Langfuse',   'http://localhost:3000'),
    ('Prometheus', 'http://localhost:9090/-/ready'),
    ('Grafana',    'http://localhost:3001/api/health'),
    ('RAG metrics','http://localhost:8100/metrics'),
]:
    try:
        r = httpx.get(url, timeout=3)
        print(f'✅ {name:11s} {url}  → {r.status_code}')
    except Exception as e:
        print(f'⚠️ {name:11s} 未启动')

## ✅ 总结：项目交付亮点

| 方向 | 核心交付 | 面试亮点 |
|------|---------|---------|
| 方向 A | Qwen3-8B QLoRA + Agentic RAG + MCP | 无侵入接入 Agent，RAG 三段式，fallback 双保险 |
| 方向 B | Qwen3-4B SFT→DPO→GRPO | 三路对比 + 自定义 reward + 角色一致性 ↑8pp |
| 工程 | FP8/GGUF/ExecuTorch/QNN/MLC 五路部署 | 云端 + 桌面 + 移动 + NPU + WebGPU |
| 观测 | Langfuse + Prometheus + Grafana | session 级端到端 trace |

📄 详见 `INTERVIEW.md`（项目速查卡）